# Escalation after the agreement rule
Step 1 (6 queries): answer if inliers ≥ 8 or ≥ 2 queries agree within 1σ (each with ≥ 3 inliers).
Step 2: photos still skipped are re-run with 27 queries (9 headings −10…+10 × scales 0.95, 1.0, 1.05; dumps `dump_esc27_*`).
The escalated decision picks the most inliers + z over all 27 queries and answers at ≥ 8 inliers or ≥ k agreeing queries.
More queries make chance agreement more likely, so k = 2, 3, 4 are all scored. Wrong answers matter most on NUDT.

In [ ]:
import sys, glob
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd()
while not ((ROOT / "src").is_dir() and (ROOT / "train/checks").is_dir()):
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from train.checks.vote_decision import load_photo, choose_max, query_support, error_sigma
from train.checks.decision_gate import points_from_error
R = ROOT / "tests/Test_Iteration_3/results/fresh_run"

def first_step(name):
    """Per photo from the 6-query dump (or the gate-row csv for NUDT seed 101): error, inliers, agreement."""
    if name == "nudt101":
        g = pd.read_csv(R / "gate_rows2_nudt_conf0.22.csv")
        return pd.DataFrame(dict(photo=g.photo, err=g.error_sigma, inl=g.inliers, agree=g.agree_queries))
    rows = []
    for f in glob.glob(str(R / ("dump_%s_conf0.22" % name) / "photo_*.npz")):
        P = load_photo(f); c = P["cand"]; k = choose_max(c); k = k if k >= 0 else int(np.argmax(c["inliers"]))
        rows.append(dict(photo=int(Path(f).stem.split("_")[1]), err=error_sigma(c["x"][k], c["y"][k], P["truth"], P["sigma"]),
                         inl=c["inliers"][k], agree=query_support(c, c["x"][k], c["y"][k], P["sigma"])))
    return pd.DataFrame(rows)

def escalated(tag):
    rows = []
    for f in glob.glob(str(R / ("dump_esc27_%s_conf0.22" % tag) / "photo_*.npz")):
        P = load_photo(f); c = P["cand"]; k = choose_max(c)
        if k < 0:
            rows.append(dict(photo=int(Path(f).stem.split("_")[1]), err2=np.inf, inl2=0, agree2=0, heading2=np.nan)); continue
        rows.append(dict(photo=int(Path(f).stem.split("_")[1]), err2=error_sigma(c["x"][k], c["y"][k], P["truth"], P["sigma"]),
                         inl2=c["inliers"][k], agree2=query_support(c, c["x"][k], c["y"][k], P["sigma"]), heading2=c["heading"][k]))
    return pd.DataFrame(rows)

SETS = {"official": "official", "nudt101": "nudt101", "nudt202": "nudt_seed202"}
results = []
for tag, dump in SETS.items():
    t = first_step(dump if tag != "nudt101" else "nudt101").merge(escalated(tag), on="photo", how="left")
    pts1 = points_from_error(t.err.values)
    bar8 = t.inl.values >= 8
    step1 = bar8 | (t.agree.values >= 2)
    def score(f, p): return np.where(f, p, 0).sum() / len(t) * 6
    def wrong(f, p): return int((f & (p == -20)).sum())
    results.append(dict(set=tag, n=len(t), escalated=int(t.err2.notna().sum()), variant="bar 8", score=score(bar8, pts1), wrong=wrong(bar8, pts1)))
    results.append(dict(set=tag, n=len(t), escalated=int(t.err2.notna().sum()), variant="agree>=2", score=score(step1, pts1), wrong=wrong(step1, pts1)))
    esc = t.err2.notna().values
    pts2 = np.where(esc, points_from_error(np.nan_to_num(t.err2.values, posinf=1e9)), pts1)
    for k in [2, 3, 4]:
        f2 = step1 | (esc & ((t.inl2.values >= 8) | (t.agree2.values >= k)))
        pts = np.where(step1, pts1, pts2)
        results.append(dict(set=tag, n=len(t), escalated=int(esc.sum()), variant="agree>=2 + esc27 (bar 8 or agree>=%d)" % k, score=score(f2, pts), wrong=wrong(f2, pts)))
    for bar2 in [10, 11, 12, 15]:
        f2 = step1 | (esc & (t.inl2.values >= bar2))
        pts = np.where(step1, pts1, pts2)
        results.append(dict(set=tag, n=len(t), escalated=int(esc.sum()), variant="agree>=2 + esc27 (bar %d only)" % bar2, score=score(f2, pts), wrong=wrong(f2, pts)))
    print(tag); display(t[esc].sort_values("inl2", ascending=False).head(25).round(2))
display(pd.DataFrame(results).round(2))

**Why the escalated bar must be higher.** With 27 queries the decision looks at 270 candidates instead of 60, so the best false place gets more chances to collect inliers. On the escalated NUDT photos the strongest false place reached 8 or more inliers on 12 % of photos (max 10); on the 6-query runs the official max false count was 14 but the true place nearly always beat it. Agreement among the 27 queries is also not independent evidence: neighbouring headings 2.5° apart see almost the same image and land on the same false place (NUDT wrong answers with 6 agreeing queries).